<a href="https://colab.research.google.com/github/pejmanrasti/SMART_MICROSCOPY/blob/main/AgenticAI/02_Coding_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Install Necessary Libraries

In [ ]:
!pip install openai

### 2. Configure OpenRouter API Connection

To keep your credentials secure, add your OpenRouter API key using Google Colab's secret manager (the key icon on the left panel) with the name `OPENROUTER_API_KEY`.

In [1]:
from google.colab import userdata
from openai import OpenAI

try:
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
except Exception:
    OPENROUTER_API_KEY = "your_openrouter_API_Key"  # Fallback

# Configure client for OpenRouter
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=OPENROUTER_API_KEY,
)

### 3. Build the Agentic Assistant Class
We define a class that takes your request, queries a model via OpenRouter (e.g., `google/gemini-2.5-flash`), and recommends suitable architectures (CNNs, Transformers, U-Net).

In [2]:
import json
import os
from pathlib import Path

class ArchitectAgent:
    def __init__(self, client):
        self.client = client
        self.model = "baidu/ernie-5.5-flash-preview:free"

    def analyze_data_path(self, path_str):
        """Inspects the provided path and returns structure insights."""
        if not path_str or not os.path.exists(path_str):
            return {"status": "error", "message": f"Path '{path_str}' does not exist or was not provided."}

        path = Path(path_str)
        subdirs = [d for d in path.iterdir() if d.is_dir()]
        subdir_names = [d.name.lower() for d in subdirs]

        # Check for Segmentation Structure
        segmentation_keywords = {'image', 'images', 'mask', 'masks', 'labels'}
        has_images = any('image' in name for name in subdir_names)
        has_masks = any('mask' in name or 'label' in name for name in subdir_names)

        if has_images and has_masks:
            return {
                "status": "success",
                "detected_task": "Segmentation",
                "message": f"Detected two main directories containing images and masks: {[d.name for d in subdirs]}. Your data structure looks perfect for Image Segmentation!"
            }

        # Check for Classification Structure (multiple subfolders corresponding to different classes)
        if len(subdirs) >= 2:
            sample_files_per_dir = {d.name: len(list(d.glob('*'))) for d in subdirs[:5]}
            return {
                "status": "success",
                "detected_task": "Classification",
                "message": f"Detected {len(subdirs)} subfolders ({[d.name for d in subdirs[:5]]}...). Your data structure looks perfect for Image Classification!"
            }

        return {
            "status": "unknown",
            "detected_task": None,
            "message": f"Found {len(subdirs)} subdirectories. We couldn't auto-classify the structure confidently. Please select your task manually."
        }

    def propose_model(self, task_type, additional_details=""):
        prompt = f"""You are an expert AI architect assisting a researcher.
They want to perform the following task: {task_type}.
Additional details / context provided: {additional_details}

Based on this, propose:
1. A custom deep learning architecture (e.g., CNN, Transformer, or U-Net-based variant) suitable for training from scratch or fine-tuning.
2. State-of-the-art Foundation Models (e.g., from Hugging Face or API-based) that can solve this out-of-the-box or with minimal adaptation.

Keep your recommendations concise, structured, and highly actionable."""

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": "You are a helpful AI assistant specializing in computer vision and deep learning model architecture selection."},
                    {"role": "user", "content": prompt}
                ]
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"An error occurred during the API call: {str(e)}"

agent = ArchitectAgent(client)

### 3.5. Mount Google Drive and Unzip Dataset (Optional)
Run this block to mount your Google Drive, locate your zipped dataset, and extract it here for the agent to inspect.

In [3]:
from google.colab import drive
# 1. Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import zipfile
import os


# 2. Define the path to your zip file on Google Drive
# Update this path to where your zip file is located on your Drive
zip_path = 'path_to_your_data'
extract_to = '/content/dataset'

# 3. Unzip the file if it exists
if os.path.exists(zip_path):
    print(f"Extracting {zip_path} to {extract_to}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print("Extraction complete! Use '/content/dataset' in the next step.")
else:
    print(f"Zip file not found at: {zip_path}. Please update the path and try again.")

### 4. Interactive Task Selection and Recommendation Loop
Run this cell to select your task, input your data constraints, and receive recommendations immediately.

In [ ]:
import subprocess
import sys

# Ensure segmentation-models-pytorch is installed silently
try:
    import segmentation_models_pytorch as smp
except ImportError:
    print("Installing segmentation-models-pytorch (smp)... ")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch"])
    import segmentation_models_pytorch as smp

import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
import torch
import torch.nn as nn
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import os
from datetime import datetime
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# UI Elements
path_input = widgets.Text(
    value='/content/dataset',
    placeholder='e.g., /content/dataset',
    description='Data Path:',
    layout=widgets.Layout(width='90%')
)

task_dropdown = widgets.Dropdown(
    options=['Classification', 'Segmentation'],
    value='Classification',
    description='Task Mode:',
)

details_input = widgets.Textarea(
    value='',
    placeholder='E.g., resnet18, mobilenet, or efficientnet for Classification; unet or fpn for Segmentation.',
    description='Model Prefer:',
    layout=widgets.Layout(width='90%', height='80px')
)

submit_btn = widgets.Button(
    description='Select, Load & Train Model',
    button_style='success',
    icon='gears'
)

output_area = widgets.Output()

# --- Classification Dataset ---
class SimpleClassificationDataset(Dataset):
    def __init__(self, data_dir, img_size=(128, 128)):
        self.data_dir = data_dir
        self.img_size = img_size
        self.classes = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

        self.samples = []
        for cls_name in self.classes:
            cls_dir = os.path.join(data_dir, cls_name)
            for root, _, files in os.walk(cls_dir):
                for f in files:
                    if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                        self.samples.append((os.path.join(root, f), self.class_to_idx[cls_name]))

        if len(self.samples) == 0:
            raise RuntimeError(f"No images found in subfolders of {data_dir}.")

        self.transform = transforms.Compose([
            transforms.Resize(self.img_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image_pil = Image.open(img_path).convert('RGB')
        image_tensor = self.transform(image_pil)
        return image_tensor, label

# --- Segmentation Dataset ---
class SimpleSegmentationDataset(Dataset):
    def __init__(self, data_dir, img_size=(128, 128)):
        self.data_dir = data_dir
        self.img_size = img_size
        self.ids = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]

        if len(self.ids) == 0:
            raise RuntimeError(f"No sample subdirectories found in {data_dir}.")

        self.transform = transforms.Compose([
            transforms.Resize(self.img_size),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        id_ = self.ids[idx]
        sample_dir = os.path.join(self.data_dir, id_)

        img_dir = os.path.join(sample_dir, 'images')
        img_file = os.listdir(img_dir)[0]
        image_pil = Image.open(os.path.join(img_dir, img_file)).convert('RGB')
        image_tensor = self.transform(image_pil)

        masks_dir = os.path.join(sample_dir, 'masks')
        mask_files = os.listdir(masks_dir)
        aggregated_mask = np.zeros((self.img_size[0], self.img_size[1]), dtype=np.float32)

        for mask_file in mask_files:
            mask_path = os.path.join(masks_dir, mask_file)
            mask_pil = Image.open(mask_path).convert('L')
            mask_resized = mask_pil.resize(self.img_size, Image.NEAREST)
            mask_arr = np.array(mask_resized, dtype=np.float32) / 255.0
            aggregated_mask = np.maximum(aggregated_mask, mask_arr)

        mask_tensor = torch.tensor(aggregated_mask).unsqueeze(0)
        return image_tensor, mask_tensor

# --- LLM Model Selector and Identifier ---
def ask_llm_for_architecture_selection(task, dataset_summary, user_details):
    """Asks the LLM to choose a model name from torchvision or smp"""
    if task == "Classification":
        prompt = f"""You are an AI Deep Learning specialist.
Based on these dataset details: {dataset_summary} and user requests: {user_details},
choose the best standard model from `torchvision.models` (e.g., 'resnet18', 'mobilenet_v3_small', 'efficientnet_b0', 'vgg11').
Respond with only a single word corresponding to the name of the function in torchvision.models (lowercase). Do not include any punctuation, quotes, or markdown code blocks."""
    else:
        prompt = f"""You are an AI Deep Learning specialist.
Based on these dataset details: {dataset_summary} and user requests: {user_details},
choose the best model class from the `segmentation_models_pytorch` library (e.g., 'Unet', 'Linknet', 'FPN', 'PSPNet').
Respond with only a single word corresponding to the class name in smp (e.g., Unet). Do not include any punctuation or markdown code blocks."""

    try:
        response = client.chat.completions.create(
            model=agent.model,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content.strip().replace("`", "")
    except Exception:
        return "resnet18" if task == "Classification" else "Unet"

def get_torchvision_model(model_name, num_classes):
    """Retrieves a classification model from torchvision and adapts the output head."""
    model_fn = getattr(models, model_name, models.resnet18)
    try:
        # Try loading with default pretrained weights
        try:
            weights_attr = next(iter([attr for attr in dir(models) if attr.lower().startswith(model_name) and 'weights' in attr.lower()]), None)
            if weights_attr:
                weights = getattr(models, weights_attr).DEFAULT
                model = model_fn(weights=weights)
            else:
                model = model_fn(pretrained=True)
        except Exception:
            model = model_fn(pretrained=True)
    except Exception:
        model = model_fn()

    # Modify classification head
    if hasattr(model, 'fc'):
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
    elif hasattr(model, 'classifier'):
        if isinstance(model.classifier, nn.Sequential):
            in_features = model.classifier[-1].in_features
            model.classifier[-1] = nn.Linear(in_features, num_classes)
        else:
            in_features = model.classifier.in_features
            model.classifier = nn.Linear(in_features, num_classes)
    elif hasattr(model, 'heads'):
        in_features = model.heads.head.in_features
        model.heads.head = nn.Linear(in_features, num_classes)
    return model

def get_smp_model(model_name):
    """Retrieves a segmentation model from segmentation_models_pytorch."""
    model_cls = getattr(smp, model_name, smp.Unet)
    return model_cls(encoder_name="resnet18", encoder_weights="imagenet", in_channels=3, classes=1)

def visualize_samples(dataset, task):
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    if task == "Classification":
        for i in range(2):
            idx = min(i * (len(dataset) // 2), len(dataset) - 1)
            img, label = dataset[idx]
            np_img = img.numpy().transpose((1, 2, 0))
            np_img = np.clip(np_img * 0.229 + 0.485, 0, 1)
            axes[i].imshow(np_img)
            class_name = dataset.classes[label] if hasattr(dataset, 'classes') else str(label)
            axes[i].set_title(f"Class: {class_name}")
            axes[i].axis('off')
    elif task == "Segmentation":
        img, mask = dataset[0]
        np_img = img.numpy().transpose((1, 2, 0))
        np_mask = mask.squeeze().numpy()
        axes[0].imshow(np_img)
        axes[0].set_title("Sample Image")
        axes[0].axis('off')
        axes[1].imshow(np_mask, cmap='gray')
        axes[1].set_title("Mask")
        axes[1].axis('off')
    plt.tight_layout()
    plt.show()

def dice_coeff(preds, targets, threshold=0.5):
    preds = (preds > threshold).float()
    targets = (targets > 0.5).float()
    intersection = (preds * targets).sum()
    total = preds.sum() + targets.sum()
    return (2. * intersection + 1e-6) / (total + 1e-6)

def execute_pipeline(task, data_path, user_details, epochs=20):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\nUsing device: {device}")

    output_folder = "/content/metrics_output"
    os.makedirs(output_folder, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")

    try:
        if task == "Classification":
            dataset = SimpleClassificationDataset(data_path)
            num_classes = len(dataset.classes)
            dataset_summary = f"Classification with {len(dataset)} items in {num_classes} classes. Input (3,128,128)"

            print("\n--- [Dataset Info] Loading and Visualizing Source Data ---")
            visualize_samples(dataset, "Classification")

            print("\nConsulting LLM for the optimal torchvision model selection...")
            selected_model_name = ask_llm_for_architecture_selection(task, dataset_summary, user_details)
            print(f"Selected Architecture: torchvision.models.{selected_model_name}")

            model = get_torchvision_model(selected_model_name, num_classes).to(device)

            n_total = len(dataset)
            n_train = int(0.7 * n_total)
            n_val = int(0.15 * n_total)
            n_test = n_total - n_train - n_val
            train_set, val_set, test_set = random_split(dataset, [n_train, n_val, n_test])

            train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
            val_loader = DataLoader(val_set, batch_size=16, shuffle=False)
            test_loader = DataLoader(test_set, batch_size=16, shuffle=False)

            criterion = nn.CrossEntropyLoss()
            optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

            train_losses, val_losses = [], []
            train_accs, val_accs = [], []

            print(f"\nTraining {selected_model_name} for {epochs} epochs...")
            for epoch in range(epochs):
                model.train()
                running_loss, correct = 0.0, 0
                for images, labels in train_loader:
                    images, labels = images.to(device), labels.to(device)
                    optimizer.zero_grad()
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                    loss.backward()
                    optimizer.step()
                    running_loss += loss.item() * images.size(0)
                    _, predicted = outputs.max(1)
                    correct += predicted.eq(labels).sum().item()

                epoch_train_loss = running_loss / len(train_set)
                epoch_train_acc = correct / len(train_set)
                train_losses.append(epoch_train_loss)
                train_accs.append(epoch_train_acc)

                model.eval()
                epoch_val_loss, val_correct = 0.0, 0
                with torch.no_grad():
                    for images, labels in val_loader:
                        images, labels = images.to(device), labels.to(device)
                        outputs = model(images)
                        loss = criterion(outputs, labels)
                        epoch_val_loss += loss.item() * images.size(0)
                        _, predicted = outputs.max(1)
                        val_correct += predicted.eq(labels).sum().item()

                epoch_val_loss /= len(val_set)
                epoch_val_acc = val_correct / len(val_set)
                val_losses.append(epoch_val_loss)
                val_accs.append(epoch_val_acc)
                print(f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_train_loss:.4f} - Val Loss: {epoch_val_loss:.4f} - Val Accuracy: {epoch_val_acc:.4f}")

            model_save_path = os.path.join(output_folder, "standard_model.pth")
            torch.save(model.state_dict(), model_save_path)

            plt.figure(figsize=(10, 4))
            plt.subplot(1, 2, 1)
            plt.plot(train_losses, label='Train')
            plt.plot(val_losses, label='Val')
            plt.title('Loss')
            plt.legend()
            plt.subplot(1, 2, 2)
            plt.plot(train_accs, label='Train')
            plt.plot(val_accs, label='Val')
            plt.title('Accuracy')
            plt.legend()
            curve_filename = f"{selected_model_name}_learning_curves_{timestamp}.png"
            plt.savefig(os.path.join(output_folder, curve_filename))
            plt.close()
            print(f"\nSaved learning curves as: {curve_filename}")

            # Test evaluation
            model.eval()
            total_test_loss, test_correct = 0.0, 0
            with torch.no_grad():
                for images, labels in test_loader:
                    images, labels = images.to(device), labels.to(device)
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                    total_test_loss += loss.item() * images.size(0)
                    _, predicted = outputs.max(1)
                    test_correct += predicted.eq(labels).sum().item()

            mean_test_loss = total_test_loss / len(test_set)
            mean_test_acc = test_correct / len(test_set)

            # Visualize Test Prediction
            test_img, test_label = test_set[0]
            with torch.no_grad():
                logits = model(test_img.unsqueeze(0).to(device))
                pred_idx = torch.argmax(logits, dim=1).item()
                probs = torch.softmax(logits, dim=1).cpu().squeeze(0).numpy()

            actual_class = dataset.classes[test_label]
            predicted_class = dataset.classes[pred_idx]

            fig, axes = plt.subplots(1, 2, figsize=(10, 4))
            np_test_img = test_img.numpy().transpose((1, 2, 0))
            np_test_img = np.clip(np_test_img * 0.229 + 0.485, 0, 1)
            axes[0].imshow(np_test_img)
            axes[0].set_title(f"Test Sample (True Class: {actual_class})")
            axes[0].axis('off')

            y_pos = np.arange(len(dataset.classes))
            axes[1].barh(y_pos, probs, color='skyblue')
            axes[1].set_yticks(y_pos)
            axes[1].set_yticklabels(dataset.classes)
            axes[1].set_xlabel('Probability')
            axes[1].set_title(f"Prediction probabilities (Predicted: {predicted_class})")
            plt.tight_layout()
            plt.show()

            result_text = f"**Final Results with standard {selected_model_name} architecture:**\n"
            result_text += f"*   Mean Test Loss: {mean_test_loss:.4f}\n"
            result_text += f"*   **Final Test Accuracy: {mean_test_acc:.4f}**\n\n"
            result_text += f"*   Model Checkpoint Saved: `{model_save_path}`\n"
            return result_text

        elif task == "Segmentation":
            dataset = SimpleSegmentationDataset(data_path)
            dataset_summary = f"Segmentation with {len(dataset)} items. Input (3,128,128)"

            print("\n--- [Dataset Info] Loading and Visualizing Source Data ---")
            visualize_samples(dataset, "Segmentation")

            print("\nConsulting LLM for the optimal SMP model selection...")
            selected_model_name = ask_llm_for_architecture_selection(task, dataset_summary, user_details)
            print(f"Selected Architecture: segmentation_models_pytorch.{selected_model_name}")

            model = get_smp_model(selected_model_name).to(device)

            n_total = len(dataset)
            n_train = int(0.7 * n_total)
            n_val = int(0.15 * n_total)
            n_test = n_total - n_train - n_val
            train_set, val_set, test_set = random_split(dataset, [n_train, n_val, n_test])

            train_loader = DataLoader(train_set, batch_size=8, shuffle=True)
            val_loader = DataLoader(val_set, batch_size=8, shuffle=False)
            test_loader = DataLoader(test_set, batch_size=8, shuffle=False)

            criterion = nn.BCEWithLogitsLoss()
            optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

            train_losses, val_losses = [], []
            train_dices, val_dices = [], []

            print(f"\nTraining SMP {selected_model_name} for {epochs} epochs...")
            for epoch in range(epochs):
                model.train()
                running_loss, running_dice = 0.0, 0.0
                for images, masks in train_loader:
                    images, masks = images.to(device), masks.to(device)
                    optimizer.zero_grad()
                    outputs = model(images)
                    loss = criterion(outputs, masks)
                    loss.backward()
                    optimizer.step()
                    running_loss += loss.item() * images.size(0)
                    running_dice += dice_coeff(torch.sigmoid(outputs), masks).item() * images.size(0)

                epoch_train_loss = running_loss / len(train_set)
                epoch_train_dice = running_dice / len(train_set)
                train_losses.append(epoch_train_loss)
                train_dices.append(epoch_train_dice)

                model.eval()
                epoch_val_loss, epoch_val_dice = 0.0, 0.0
                with torch.no_grad():
                    for images, masks in val_loader:
                        images, masks = images.to(device), masks.to(device)
                        outputs = model(images)
                        loss = criterion(outputs, masks)
                        epoch_val_loss += loss.item() * images.size(0)
                        epoch_val_dice += dice_coeff(torch.sigmoid(outputs), masks).item() * images.size(0)

                epoch_val_loss /= len(val_set)
                epoch_val_dice /= len(val_set)
                val_losses.append(epoch_val_loss)
                val_dices.append(epoch_val_dice)
                print(f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_train_loss:.4f} - Val Loss: {epoch_val_loss:.4f} - Val Dice: {epoch_val_dice:.4f}")

            model_save_path = os.path.join(output_folder, "standard_model.pth")
            torch.save(model.state_dict(), model_save_path)

            # Save Training Curves for Segmentation
            plt.figure(figsize=(10, 4))
            plt.subplot(1, 2, 1)
            plt.plot(train_losses, label='Train')
            plt.plot(val_losses, label='Val')
            plt.title('Loss')
            plt.legend()
            plt.subplot(1, 2, 2)
            plt.plot(train_dices, label='Train')
            plt.plot(val_dices, label='Val')
            plt.title('Dice Coefficient')
            plt.legend()
            curve_filename = f"{selected_model_name}_learning_curves_{timestamp}.png"
            plt.savefig(os.path.join(output_folder, curve_filename))
            plt.close()
            print(f"\nSaved learning curves as: {curve_filename}")

            # Evaluate on Test Set
            model.eval()
            total_test_loss, total_test_dice = 0.0, 0.0
            with torch.no_grad():
                for images, masks in test_loader:
                    images, masks = images.to(device), masks.to(device)
                    outputs = model(images)
                    loss = criterion(outputs, masks)
                    total_test_loss += loss.item() * images.size(0)
                    total_test_dice += dice_coeff(torch.sigmoid(outputs), masks).item() * images.size(0)

            mean_test_loss = total_test_loss / len(test_set)
            mean_test_dice = total_test_dice / len(test_set)

            # Visualize Test Prediction
            test_img, test_mask = test_set[0]
            with torch.no_grad():
                pred_mask = torch.sigmoid(model(test_img.unsqueeze(0).to(device))).cpu().squeeze(0)

            fig, axes = plt.subplots(1, 3, figsize=(12, 4))
            axes[0].imshow(test_img.numpy().transpose((1, 2, 0)))
            axes[0].set_title("Test Input")
            axes[0].axis('off')
            axes[1].imshow(test_mask.squeeze().numpy(), cmap='gray')
            axes[1].set_title("Ground Truth")
            axes[1].axis('off')
            im_pred = axes[2].imshow(pred_mask.squeeze().numpy(), cmap='hot', vmin=0, vmax=1)
            axes[2].set_title(f"{selected_model_name} Prediction Map")
            axes[2].axis('off')
            fig.colorbar(im_pred, ax=axes[2], shrink=0.8)
            plt.tight_layout()
            plt.show()

            result_text = f"**Final Results with SMP {selected_model_name} architecture:**\n"
            result_text += f"*   Mean Test Loss: {mean_test_loss:.4f}\n"
            result_text += f"*   **Final Test Dice Score: {mean_test_dice:.4f}**\n\n"
            result_text += f"*   Model Checkpoint Saved: `{model_save_path}`\n"
            return result_text

    except Exception as e:
        return f"An error occurred during training execution: {str(e)}"

def on_button_click(b):
    with output_area:
        clear_output()
        path_val = path_input.value.strip()
        task_mode = task_dropdown.value
        details_val = details_input.value.strip()

        print("Running training pipeline...")
        pipeline_outcome = execute_pipeline(task_mode, path_val, details_val, epochs=20)
        display(Markdown(f"### Execution Results\n{pipeline_outcome}"))

submit_btn.on_click(on_button_click)

# Render dynamic execution panel
display(Markdown("### Standard Prebuilt & SMP Library Model Selector"))
display(path_input)
display(task_dropdown)
display(details_input)
display(submit_btn)
display(output_area)

### 🛠️ Practical Work (TP): Customizing and Optimizing Your Pipeline

**Duration:** 30–40 minutes  
**Objective:** Improve your model's performance on the dataset by introducing modifications to the training setup, model hyperparameters, and the LLM recommendation engine.

---

### 📝 Your Tasks:


#### **Task 1: Experiment with a Different LLM Backend (15 mins) ⭐**
*   Our recommender agent currently uses `baidu/ernie-5.5-flash-preview:free` to decide the best prebuilt architecture.
*   **Your Mission:** Modify the LLM model identifier string in the initialization of the `ArchitectAgent` (in **Section 3: Build the Agentic Assistant Class**, inside the `__init__` constructor).
*   *Hint:* Re-run the cell defining the `ArchitectAgent` and then trigger the interactive panel. See if the recommended model for your dataset changes!

#### **Task 2: Change the Model Architecture & Hyperparameters (10 mins)**
*   **Classification:** In the interactive panel above, type `efficientnet_b0` or `mobilenet_v3_large` into **Model Prefer**. Change the number of epochs to `15` or `20`.
*   **Segmentation:** Type `Linknet` or `FPN` in the **Model Prefer** input.
*   *Hint:* Observe how the training speed and initial losses differ between networks.

#### **Task 3: Implement Learning Rate Decay or Optimizer Tweak (10 mins)**
*   Currently, we use the `Adam` optimizer with a static learning rate of `1e-4`.
*   **Your Mission:** Experiment with changing the optimizer to `SGD` with momentum, or implement a basic learning rate scheduler (e.g., `torch.optim.lr_scheduler.StepLR`) to lower the learning rate as epochs progress.
*   *Hint:* Scroll down to the training loop inside `execute_pipeline`, add your scheduler, and remember to call `scheduler.step()` at the end of each epoch.